Fixed GridSearchCV package conflict by switching to local sklearn grid search
*Co-authored with CoCo*

# Donor Lapse/Churn Intelligence — (2 of 3) Model: Training, Experiments, HPO & Registry

**Notebook 2 of 3.** Trains a custom **XGBoost** lapse classifier on the Feature-Store-backed dataset from notebook 01, tracks every run with **Experiment Tracking**, tunes hyperparameters, optionally trains as a remote **ML Job**, then logs the model to the **Model Registry** with explainability enabled.

> Run **`donor-churn-01-features.ipynb`** first — this notebook loads the `LAPSE_TRAINING_SET/v1` dataset it produced.

---
## Rehydrate session & objects

This notebook is standalone: it reconnects and loads the persisted Feature Store / Dataset from notebook 01.

In [1]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.ml.feature_store import FeatureStore, CreationMode
from snowflake.ml.dataset import load_dataset

try:
    session = get_active_session()
except Exception:
    from snowflake.snowpark import Session
    session = Session.builder.getOrCreate()  # local run: uses your default connections.toml connection
for stmt in ["USE ROLE ACCOUNTADMIN", "USE DATABASE DONOR_CHURN_ML_DEMO",
             "USE SCHEMA MODELS", "USE WAREHOUSE DONOR_CHURN_ML_WH"]:
    session.sql(stmt).collect()

fs = FeatureStore(session=session, database="DONOR_CHURN_ML_DEMO", name="FEATURES",
                  default_warehouse="DONOR_CHURN_ML_WH", creation_mode=CreationMode.FAIL_IF_NOT_EXIST)

training_dataset = load_dataset(session, "DONOR_CHURN_ML_DEMO.FEATURES.LAPSE_TRAINING_SET", "v1")
train_sp_df = training_dataset.read.to_snowpark_dataframe()
print("Loaded LAPSE_TRAINING_SET/v1 — rows:", train_sp_df.count())

Loaded LAPSE_TRAINING_SET/v1 — rows: 50000


---
## Section 5: Snowpark ML modeling + Experiment Tracking + HPO

Train an **XGBoost** classifier with the `snowflake.ml.modeling` API (the fit runs in Snowflake — no data leaves). Every candidate configuration is logged as a **run** under one experiment, so you can compare them side-by-side in **Snowsight → AI & ML → Experiments**. We evaluate on the probability output (`predict_proba`) so AUC is meaningful.

In [2]:
FEATURES = [
    "WEALTH_SIGNAL_SCORE", "TENURE_DAYS", "FREQUENCY_LIFETIME", "FREQUENCY_LAST_12M",
    "MONETARY_TOTAL", "AVG_GIFT_AMOUNT", "RECENCY_DAYS", "ENGAGEMENT_COUNT", "ENGAGEMENT_LAST_6M",
]
LABEL = "IS_LAPSED"

train_df, val_df = train_sp_df.random_split([0.8, 0.2], seed=42)
print("train:", train_df.count(), "val:", val_df.count())

train: 39940 val: 10060


In [3]:
# Helper: given a scored Snowpark df, return the positive-class probability column name.
def positive_proba_col(scored_df):
    proba_cols = [c for c in scored_df.columns if "PROBA" in c.upper()]
    if not proba_cols:
        raise ValueError("No predict_proba output columns found: " + str(scored_df.columns))
    for c in proba_cols:
        if c.upper().rstrip('"').endswith("1"):
            return c
    return proba_cols[-1]

In [4]:
from snowflake.ml.modeling.xgboost import XGBClassifier
from snowflake.ml.modeling.metrics import roc_auc_score, f1_score
from snowflake.ml.experiment import ExperimentTracking

exp = ExperimentTracking(session=session, database_name="DONOR_CHURN_ML_DEMO", schema_name="MODELS")
exp.set_experiment("DONOR_LAPSE")

# A few candidate configs — each becomes a tracked run.
candidates = [
    {"n_estimators": 200, "max_depth": 4, "learning_rate": 0.10},
    {"n_estimators": 300, "max_depth": 6, "learning_rate": 0.05},
    {"n_estimators": 400, "max_depth": 6, "learning_rate": 0.05},
]

results = []
for i, params in enumerate(candidates):
    with exp.start_run(f"xgb_candidate_{i+1}"):
        exp.log_params(params)
        m = XGBClassifier(input_cols=FEATURES, label_cols=[LABEL], output_cols=["PRED_LAPSE"], **params)
        m.fit(train_df)
        scored = m.predict_proba(val_df)
        pcol = positive_proba_col(scored)
        auc = roc_auc_score(df=scored, y_true_col_names=LABEL, y_score_col_names=pcol)
        preds = m.predict(val_df)
        f1 = f1_score(df=preds, y_true_col_names=LABEL, y_pred_col_names="PRED_LAPSE")
        exp.log_metrics({"val_auc": float(auc), "val_f1": float(f1)})
        results.append((params, float(auc), float(f1)))
        print(f"run {i+1}: params={params} AUC={auc:.3f} F1={f1:.3f}")

best_params, best_auc, best_f1 = max(results, key=lambda r: r[1])
print("Best config:", best_params, "AUC=", round(best_auc, 3))

/Users/snuggehalli/Documents/Internal_Demos/field-demo-enablement/donor-churn-ml/.venv-ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Package 'snowflake-telemetry-python' is not installed in the local environment. Your UDF might not work when the package is installed on the server but not on your local environment.


/Users/snuggehalli/Documents/Internal_Demos/field-demo-enablement/donor-churn-ml/.venv-ml/lib/python3.11/site-packages/snowflake/ml/model/model_signature.py:74: UserWarning: The sample input has 39940 rows. Using the first 100 rows to define the inputs and outputs of the model and the data types of each. Use `signatures` parameter to specify model inputs and outputs manually if the automatic inference is not correct.
  warnings.warn(


/Users/snuggehalli/Documents/Internal_Demos/field-demo-enablement/donor-churn-ml/.venv-ml/lib/python3.11/site-packages/snowflake/ml/model/model_signature.py:74: UserWarning: The sample input has 39940 rows. Using the first 100 rows to define the inputs and outputs of the model and the data types of each. Use `signatures` parameter to specify model inputs and outputs manually if the automatic inference is not correct.
  warnings.warn(


DataFrame.flatten() is deprecated since 0.7.0. Use `DataFrame.join_table_function()` instead.


...

Package 'snowflake-telemetry-python' is not installed in the local environment. Your UDF might not work when the package is installed on the server but not on your local environment.


/Users/snuggehalli/Documents/Internal_Demos/field-demo-enablement/donor-churn-ml/.venv-ml/lib/python3.11/site-packages/snowflake/ml/model/model_signature.py:74: UserWarning: The sample input has 39940 rows. Using the first 100 rows to define the inputs and outputs of the model and the data types of each. Use `signatures` parameter to specify model inputs and outputs manually if the automatic inference is not correct.
  warnings.warn(


/Users/snuggehalli/Documents/Internal_Demos/field-demo-enablement/donor-churn-ml/.venv-ml/lib/python3.11/site-packages/snowflake/ml/model/model_signature.py:74: UserWarning: The sample input has 39940 rows. Using the first 100 rows to define the inputs and outputs of the model and the data types of each. Use `signatures` parameter to specify model inputs and outputs manually if the automatic inference is not correct.
  warnings.warn(


run 3: params={'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.05} AUC=0.964 F1=0.938


🏃 View run XGB_CANDIDATE_3 at: https://app.snowflake.com/_deeplink/#/experiments/databases/DONOR_CHURN_ML_DEMO/schemas/MODELS/experiments/DONOR_LAPSE/runs/XGB_CANDIDATE_3
🧪 View experiment at: https://app.snowflake.com/_deeplink/#/experiments/databases/DONOR_CHURN_ML_DEMO/schemas/MODELS/experiments/DONOR_LAPSE
Best config: {'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.1} AUC= 0.966


In [5]:
# Distributed hyperparameter search over a small grid.
# Using sklearn GridSearchCV locally to avoid server-side package conflicts.
from sklearn.model_selection import GridSearchCV as SkGridSearchCV
from xgboost import XGBClassifier as SkXGB
import pandas as pd

# Pull training data to pandas for local grid search
train_pdf = train_df.to_pandas()
X_train = train_pdf[FEATURES]
y_train = train_pdf[LABEL]

sk_grid = SkGridSearchCV(
    estimator=SkXGB(eval_metric="logloss"),
    param_grid={"n_estimators": [200, 400], "max_depth": [4, 6], "learning_rate": [0.05, 0.1]},
    scoring="f1",
    cv=3,
    n_jobs=-1,
)
sk_grid.fit(X_train, y_train)
grid_best = sk_grid.best_params_
print("GridSearchCV best params:", grid_best)

with exp.start_run("gridsearch_best"):
    exp.log_params(grid_best)
    exp.log_param("search", "GridSearchCV")

GridSearchCV best params: {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 200}


🏃 View run GRIDSEARCH_BEST at: https://app.snowflake.com/_deeplink/#/experiments/databases/DONOR_CHURN_ML_DEMO/schemas/MODELS/experiments/DONOR_LAPSE/runs/GRIDSEARCH_BEST
🧪 View experiment at: https://app.snowflake.com/_deeplink/#/experiments/databases/DONOR_CHURN_ML_DEMO/schemas/MODELS/experiments/DONOR_LAPSE


In [6]:
# Refit the FINAL model on the best params found (prefer the grid result).
final_params = {"n_estimators": int(grid_best.get("n_estimators", best_params["n_estimators"])),
                "max_depth": int(grid_best.get("max_depth", best_params["max_depth"])),
                "learning_rate": float(grid_best.get("learning_rate", best_params["learning_rate"]))}
final_model = XGBClassifier(input_cols=FEATURES, label_cols=[LABEL], output_cols=["PRED_LAPSE"], **final_params)
final_model.fit(train_df)

scored = final_model.predict_proba(val_df)
pcol = positive_proba_col(scored)
final_auc = float(roc_auc_score(df=scored, y_true_col_names=LABEL, y_score_col_names=pcol))
final_f1 = float(f1_score(df=final_model.predict(val_df), y_true_col_names=LABEL, y_pred_col_names="PRED_LAPSE"))
print(f"Final model: params={final_params}  AUC={final_auc:.3f}  F1={final_f1:.3f}")

Package 'snowflake-telemetry-python' is not installed in the local environment. Your UDF might not work when the package is installed on the server but not on your local environment.


/Users/snuggehalli/Documents/Internal_Demos/field-demo-enablement/donor-churn-ml/.venv-ml/lib/python3.11/site-packages/snowflake/ml/model/model_signature.py:74: UserWarning: The sample input has 39940 rows. Using the first 100 rows to define the inputs and outputs of the model and the data types of each. Use `signatures` parameter to specify model inputs and outputs manually if the automatic inference is not correct.
  warnings.warn(


/Users/snuggehalli/Documents/Internal_Demos/field-demo-enablement/donor-churn-ml/.venv-ml/lib/python3.11/site-packages/snowflake/ml/model/model_signature.py:74: UserWarning: The sample input has 39940 rows. Using the first 100 rows to define the inputs and outputs of the model and the data types of each. Use `signatures` parameter to specify model inputs and outputs manually if the automatic inference is not correct.
  warnings.warn(


Final model: params={'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.05}  AUC=0.967  F1=0.938


---
## Section 6: ML Jobs on Container Runtime (optional)

Productionize training as a **remote ML Job** on a Snowflake **compute pool** (Container Runtime). The `@remote` decorator ships the function to Snowflake compute; add a GPU instance family for deep models. Requires a compute pool — skipped automatically if your role can't create one.

In [7]:
# Create the compute pool + payload stage if your role has the privilege.
try:
    session.sql("""
        CREATE COMPUTE POOL IF NOT EXISTS DONOR_CHURN_ML_POOL
          MIN_NODES = 1 MAX_NODES = 2 INSTANCE_FAMILY = CPU_X64_M
    """).collect()
    print("Compute pool ready: DONOR_CHURN_ML_POOL")
except Exception as e:
    print("Skipping compute-pool creation (needs CREATE COMPUTE POOL privilege):", e)

session.sql("CREATE STAGE IF NOT EXISTS DONOR_CHURN_ML_DEMO.MODELS.PAYLOAD_STAGE").collect()

Compute pool ready: DONOR_CHURN_ML_POOL


[Row(status='PAYLOAD_STAGE already exists, statement succeeded.')]

In [8]:
from snowflake.ml.jobs import remote

@remote("DONOR_CHURN_ML_POOL", stage_name="DONOR_CHURN_ML_DEMO.MODELS.PAYLOAD_STAGE", session=session)
def train_lapse_model(dataset_name: str, dataset_version: str):
    from snowflake.snowpark.context import get_active_session
    from snowflake.ml.dataset import load_dataset
    from snowflake.ml.modeling.xgboost import XGBClassifier
    s = get_active_session()
    ds = load_dataset(s, dataset_name, dataset_version)
    df = ds.read.to_snowpark_dataframe()
    feats = ["WEALTH_SIGNAL_SCORE","TENURE_DAYS","FREQUENCY_LIFETIME","FREQUENCY_LAST_12M",
             "MONETARY_TOTAL","AVG_GIFT_AMOUNT","RECENCY_DAYS","ENGAGEMENT_COUNT","ENGAGEMENT_LAST_6M"]
    m = XGBClassifier(input_cols=feats, label_cols=["IS_LAPSED"], output_cols=["PRED_LAPSE"],
                      n_estimators=400, max_depth=6, learning_rate=0.05)
    m.fit(df)
    return m

try:
    job = train_lapse_model("DONOR_CHURN_ML_DEMO.FEATURES.LAPSE_TRAINING_SET", "v1")
    print("Job:", job.id, "status:", job.status)
    _ = job.result()
    print("Remote training complete (using the local final_model for registration below).")
except Exception as e:
    print("ML Job needs a compute pool + snowflake-ml-python>=1.26. Using the local model instead.", e)

Job: DONOR_CHURN_ML_DEMO.MODELS.TRAIN_LAPSE_MODEL_DD959549_1C987SIBJIW6W status: PENDING


Remote training complete (using the local final_model for registration below).


---
## Section 7: Model Registry

Log the trained model as a **versioned, governed object** with metrics, signature, sample input, and **explainability enabled**, then promote to **DEFAULT**. We also attach the model to the experiment run for full lineage.

In [9]:
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name="DONOR_CHURN_ML_DEMO", schema_name="MODELS")

mv = reg.log_model(
    model=final_model,
    model_name="DONOR_LAPSE_MODEL",
    version_name="v1",
    metrics={"auc": final_auc, "f1": final_f1},
    sample_input_data=train_df.select(FEATURES).limit(100),
    comment="XGBoost donor lapse classifier (Feature-Store RFM + engagement + wealth features)",
    options={"enable_explainability": True},
    target_platforms=["WAREHOUSE"],
)
print("Logged DONOR_LAPSE_MODEL/v1")
mv.show_functions()

reg.get_model("DONOR_LAPSE_MODEL").default = "v1"
print("DEFAULT version set to v1")

Logging model:   0%|          | 0/6 [00:00<?, ?it/s]

Logging model: validating model and dependencies...:   0%|          | 0/6 [00:00<?, ?it/s]

Logging model: packaging model...:  17%|█▋        | 1/6 [00:02<00:10,  2.09s/it]          

Logging model: packaging model...:  33%|███▎      | 2/6 [00:02<00:04,  1.05s/it]

Logging model: creating model manifest...:  33%|███▎      | 2/6 [00:02<00:04,  1.05s/it]

...

Logging model: model logged successfully!: 100%|██████████| 6/6 [00:18<00:00,  3.04s/it]

Model logged successfully.: 100%|██████████| 6/6 [00:18<00:00,  3.04s/it]               

Model logged successfully.: 100%|██████████| 6/6 [00:18<00:00,  3.10s/it]


Logged DONOR_LAPSE_MODEL/v1


DEFAULT version set to v1


---
## Section 8: Model Explainability

Because we logged with `enable_explainability=True`, the registered model exposes an **`explain`** function that returns **Shapley attributions** — so fundraisers see *why* a donor is flagged at risk, with no separate tooling.

In [10]:
explain_input = val_df.select(FEATURES).limit(10)
shap_df = mv.run(explain_input, function_name="explain")
shap_df.show()

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"WEALTH_SIGNAL_SCORE"  |"TENURE_DAYS"  |"FREQUENCY_LIFETIME"  |"FREQUENCY_LAST_12M"  |"MONETARY_TOTAL"    |"AVG_GIFT_AMOUNT"  |"RECENCY_DAYS"  |"ENGAGEMENT_COUNT"  |"ENGAGEMENT_LAST_6M"  |"""WEALTH_SIGNAL_SCORE_explanation"""  |"""TENURE_DAYS_explanation"""  |"""FREQUENCY_LIFETIME_explanation"""  |"""FREQUENCY_LAST_12M_explanation"""  |"""MONETARY_TOTAL_explanation"""  |"""AVG_GIFT_AMOUNT_explanation"""  |"""RECENCY_DAYS_explanation"""  |"""ENGAGEMENT_COUNT_explanation"""  |

---
## Notebook 2 complete

| Object | Name |
|---|---|
| Experiment | `MODELS.DONOR_LAPSE` (multiple tracked runs) |
| Registered model | `DONOR_LAPSE_MODEL/v1` (DEFAULT), `enable_explainability=True` |

**Next:** open **`donor-churn-03-serve-agent.ipynb`** to score donors, monitor drift, wrap the model in tools, and build the agent.